In [ ]:
import numpy as np
from freshflow.loader import load_sales, save_array
from freshflow.metrics import build_metrics
from freshflow.trends import daily_totals, daily_change, daily_pct_change, rolling_average, unusual_changes

clean = load_sales("../data/processed/freshflow_clean.csv", expected_rows=None)
metrics = build_metrics(clean)
revenue, transactions = daily_totals(clean, metrics)

In [ ]:
change = np.concatenate(([np.nan], daily_change(revenue)))
pct = np.concatenate(([np.nan], daily_pct_change(revenue)))
average = rolling_average(revenue, 7)
cumulative = np.cumsum(revenue)

print("day  transactions     revenue    change   change%   7-day avg   cumulative")
for day in range(30):
    print(f"{day + 1:3}  {transactions[day]:12}  {revenue[day]:10,.0f}  {change[day]:8,.0f}  {pct[day]:8.1f}  {average[day]:10,.0f}  {cumulative[day]:11,.0f}")

In [ ]:
print("Highest revenue day:", revenue.argmax() + 1, round(revenue.max(), 2))
print("Lowest revenue day:", revenue.argmin() + 1, round(revenue.min(), 2))
print("Total:", round(revenue.sum(), 2))
print("Average days 1-15:", round(revenue[:15].mean(), 2))
print("Average days 16-30:", round(revenue[15:].mean(), 2))

In [ ]:
z_scores, unusual = unusual_changes(revenue)
print("Unusual days (|z| > 2):", np.nonzero(unusual)[0] + 2)

for position in np.argsort(np.abs(z_scores))[::-1][:5]:
    print("day", position + 2, "change", round(daily_change(revenue)[position]), "z", round(z_scores[position], 2))

In [ ]:
from freshflow.anomalies import anomaly_masks, reason_codes, reason_text, exception_register, REGISTER_COLUMNS

masks = anomaly_masks(clean, metrics)
names = list(masks)

for name, mask in masks.items():
    print(f"{name:26} {mask.sum()}")

codes = reason_codes(masks)
print("Unique flagged rows:", (codes > 0).sum())

In [ ]:
values, counts = np.unique(codes[codes > 0], return_counts=True)
for value, count in zip(values, counts):
    print(count, "rows:", reason_text(int(value), names))

In [ ]:
register = exception_register(clean, metrics, masks)
print(REGISTER_COLUMNS)
for row in register[:10]:
    print(np.round(row[:9], 2), reason_text(int(row[9]), names))

In [ ]:
high_discount = masks["high discount"]
print(clean[high_discount][:, [0, 1, 2, 3, 5, 6, 8]])
print("Discount given on these rows:", round(metrics["discount_amount"][high_discount].sum(), 2))
print("Total discount given:", round(np.nansum(metrics["discount_amount"]), 2))

In [ ]:
zero_stock = masks["zero stock"]
print(clean[zero_stock][:, [0, 1, 2, 3, 5, 9]])
print("Net revenue on these rows:", round(np.nansum(metrics["net_revenue"][zero_stock]), 2))

In [ ]:
save_array("../data/processed/freshflow_exception_register.csv", register, REGISTER_COLUMNS)